## Imports

In [1]:
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from typing import Tuple, List
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import StratifiedKFold
# Check torchvision version and import accordingly
import torchvision
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

# Try different import methods for DeiT-Small
try:
    from torchvision.models import deit_small_patch16_224, DeiT_Small_Patch16_224_Weights
    print("✓ DeiT-Small imported from torchvision.models")
except ImportError:
    try:
        # Alternative: use timm library
        import timm
        print("✓ Using timm library for DeiT-Small")
        HAS_TIMM = True
    except ImportError:
        print("⚠ Warning: Neither torchvision DeiT nor timm is available")
        print("Installing timm...")
        import subprocess
        subprocess.check_call(['pip', 'install', 'timm'])
        import timm
        HAS_TIMM = True



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/anaconda3/envs/CSE499A_env/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/anaconda3/envs/CSE499A_env/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/anaconda3/envs/CSE499A_env/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/envs/CSE499A_env/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launc

PyTorch version: 2.2.2
Torchvision version: 0.17.2
✓ Using timm library for DeiT-Small


##CONFIGURATION - ALL HYPERPARAMETERS

In [2]:
DATA_ROOT = Path("/Users/alimran/Desktop/anthracnose_model_ablation/Dataset_split")  
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0
EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 5e-4
DROPOUT = 0.3
SEED = 250
LOSS_WEIGHT_HEALTH = 1.0
USE_AMP = True
PATIENCE = 5

# Model Config
NUM_SPECIES = 3
NUM_HEALTH = 2
PRETRAINED = False

# Data Augmentation Config
RESIZE_INTERPOLATION = transforms.InterpolationMode.BILINEAR
RANDOM_FLIP_PROB = 0.5
RANDOM_ROTATION_DEGREES = 15
COLOR_JITTER_BRIGHTNESS = 0.2
COLOR_JITTER_CONTRAST = 0.2
COLOR_JITTER_SATURATION = 0.2

# Normalization (ImageNet stats)
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

# Visualization Config
MAX_SAMPLE_IMAGES = 20
CONFUSION_MATRIX_DPI = 150
PLOT_DPI = 150

# Model name

N_FOLDS = 5
STATE_FILE = "kfold_state.json"

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Device: cpu


## LABEL MAPS

In [3]:
SPECIES_MAP = {"guava": 0, "mango": 1, "papaya": 2}
HEALTH_MAP = {"healthy": 0, "anthracnose": 1}

def parse_joint_label(folder_name: str) -> Tuple[int, int]:
    """Parse folder name into species and health IDs (case-insensitive)"""
    name = folder_name.strip()
    if "_" not in name:
        raise ValueError(f"Folder name not joint label: {name}")
    sp, he = name.split("_", 1)
    
    # Convert to lowercase for matching
    sp_lower = sp.lower()
    he_lower = he.lower()
    
    if sp_lower not in SPECIES_MAP:
        raise KeyError(f"Unknown species: {sp} (available: {list(SPECIES_MAP.keys())})")
    if he_lower not in HEALTH_MAP:
        raise KeyError(f"Unknown health status: {he} (available: {list(HEALTH_MAP.keys())})")
    
    sp_id = SPECIES_MAP[sp_lower]
    he_id = HEALTH_MAP[he_lower]
    return sp_id, he_id

## DATASET

In [4]:
class JointLeafDataset(Dataset):
    def __init__(self, split_root: Path, transform=None):
        self.split_root = Path(split_root)
        self.samples: List[Tuple[str, int, int]] = []
        self.transform = transform
        
        # Check if split_root exists
        if not self.split_root.exists():
            raise RuntimeError(
                f"Directory does not exist: {split_root}\n"
                f"Please check if the path is correct."
            )
        
        # Get all subdirectories
        subdirs = [d for d in self.split_root.iterdir() if d.is_dir()]
        
        if len(subdirs) == 0:
            raise RuntimeError(
                f"No subdirectories found in: {split_root}\n"
                f"Expected structure: {split_root}/species_health/images.jpg\n"
                f"Example: {split_root}/guava_healthy/img001.jpg"
            )
        
        print(f"   Found {len(subdirs)} subdirectories in {split_root.name}")
        
        for folder in sorted(subdirs):
            try:
                sp_id, he_id = parse_joint_label(folder.name)
            except (ValueError, KeyError) as e:
                print(f"   ⚠ Skipping folder '{folder.name}': {e}")
                continue
            
            # Find images
            images_in_folder = []
            for p in folder.rglob("*"):
                if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                    images_in_folder.append(str(p))
            
            print(f"   - {folder.name}: {len(images_in_folder)} images")
            
            for img_path in images_in_folder:
                self.samples.append((img_path, sp_id, he_id))
        
        if len(self.samples) == 0:
            raise RuntimeError(
                f"No images found under {split_root}\n"
                f"Found directories: {[d.name for d in subdirs]}\n"
                f"Expected directory names: species_health (e.g., 'guava_healthy' or 'Guava_Healthy')\n"
                f"Supported image formats: .jpg, .jpeg, .png, .bmp\n"
                f"Please check:\n"
                f"  1. Directory names follow 'species_health' format\n"
                f"  2. Images are in correct format\n"
                f"  3. Images are inside the subdirectories"
            )
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, sp_id, he_id = self.samples[idx]
        
        try:
            img = Image.open(path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {path}: {e}")
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        
        if self.transform is not None:
            img = self.transform(img)
        
        return img, torch.tensor(sp_id, dtype=torch.long), torch.tensor(he_id, dtype=torch.long)


class FlatSampleDataset(Dataset):
    """Wraps a plain list of (image_path, label_a, label_b) tuples with a transform."""
    def __init__(self, samples: List[Tuple[str, int, int]], transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, sp_id, he_id = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        if self.transform is not None:
            img = self.transform(img)
        return img, torch.tensor(sp_id, dtype=torch.long), torch.tensor(he_id, dtype=torch.long)

## TRANSFORMS

In [5]:
def _pil_to_tensor(img):
    """Convert PIL Image to float tensor [0,1] without numpy (avoids ABI issues)."""
    if img.mode != 'RGB':
        img = img.convert('RGB')
    w, h = img.size
    t = torch.frombuffer(bytearray(img.tobytes()), dtype=torch.uint8)
    t = t.view(h, w, 3).float().div(255.0)
    return t.permute(2, 0, 1).contiguous()


train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=RESIZE_INTERPOLATION),
    transforms.RandomHorizontalFlip(p=RANDOM_FLIP_PROB),
    transforms.RandomRotation(RANDOM_ROTATION_DEGREES),
    transforms.ColorJitter(
        brightness=COLOR_JITTER_BRIGHTNESS,
        contrast=COLOR_JITTER_CONTRAST,
        saturation=COLOR_JITTER_SATURATION
    ),
    transforms.Lambda(_pil_to_tensor),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=RESIZE_INTERPOLATION),
    transforms.Lambda(_pil_to_tensor),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])

## DATASETS & LOADERS

In [6]:
print("\n" + "="*80)
print("Loading datasets...")
print("="*80)

# Load train + val directories into a combined pool (no transform — raw path list)
_train_ds = JointLeafDataset(DATA_ROOT / "train", transform=None)
print()
_val_ds = JointLeafDataset(DATA_ROOT / "val", transform=None)
all_samples = _train_ds.samples + _val_ds.samples  # list of (path, sp_id, he_id)

# Fixed hold-out test set (never used for training or validation)
print()
test_ds = JointLeafDataset(DATA_ROOT / "test", transform=eval_tf)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

# Combined integer label for StratifiedKFold
strat_labels = [sp * NUM_HEALTH + he for (_, sp, he) in all_samples]

print("\n" + "="*80)
print(f"Combined pool (train+val): {len(all_samples)} images")
print(f"Test set (fixed hold-out): {len(test_ds)} images")
print(f"Unique strat labels: {sorted(set(strat_labels))}")
print("="*80)


Loading datasets...
   Found 6 subdirectories in train
   - Guava_Anthracnose: 1000 images
   - Guava_Healthy: 1000 images
   - Mango_Anthracnose: 1013 images
   - Mango_Healthy: 1000 images
   - Papaya_Anthracnose: 1028 images
   - Papaya_Healthy: 1000 images

   Found 6 subdirectories in val
   - Guava_Anthracnose: 35 images
   - Guava_Healthy: 199 images
   - Mango_Anthracnose: 163 images
   - Mango_Healthy: 165 images
   - Papaya_Anthracnose: 84 images
   - Papaya_Healthy: 115 images

   Found 6 subdirectories in test
   - Guava_Anthracnose: 37 images
   - Guava_Healthy: 188 images
   - Mango_Anthracnose: 165 images
   - Mango_Healthy: 165 images
   - Papaya_Anthracnose: 86 images
   - Papaya_Healthy: 127 images

Combined pool (train+val): 6802 images
Test set (fixed hold-out): 768 images
Unique strat labels: [0, 1, 2, 3, 4, 5]


## MODEL DEFINITION

In [7]:
class MultiTaskDeiTSmall(nn.Module):
    def __init__(self, num_species=NUM_SPECIES, num_health=NUM_HEALTH, 
                 pretrained=PRETRAINED, dropout=DROPOUT):
        super().__init__()
        
        # Try torchvision first, fallback to timm
        try:
            from torchvision.models import deit_small_patch16_224, DeiT_Small_Patch16_224_Weights
            # Using torchvision
            if pretrained:
                weights = DeiT_Small_Patch16_224_Weights.IMAGENET1K_V1
                self.backbone = deit_small_patch16_224(weights=weights)
            else:
                self.backbone = deit_small_patch16_224(weights=None)
            
            # Get input dimension from heads
            if isinstance(self.backbone.heads, nn.Sequential):
                in_dim = self.backbone.heads[-1].in_features
            else:
                in_dim = self.backbone.heads.in_features
            
            self.backbone.heads = nn.Identity()
            print("✓ Using DeiT-Small from torchvision")
            
        except (ImportError, AttributeError):
            # Using timm library
            import timm
            model_name = 'deit_small_patch16_224.fb_in1k' if pretrained else 'deit_small_patch16_224'
            self.backbone = timm.create_model(
                model_name,
                pretrained=pretrained,
                num_classes=0  # Remove classification head
            )
            in_dim = self.backbone.num_features
            print("✓ Using DeiT-Small from timm library")
        
        self.dropout = nn.Dropout(dropout)
        self.head_species = nn.Linear(in_dim, num_species)
        self.head_health = nn.Linear(in_dim, num_health)
    
    def forward(self, x):
        feats = self.backbone(x)
        feats = self.dropout(feats)
        logits_species = self.head_species(feats)
        logits_health = self.head_health(feats)
        return logits_species, logits_health
print("✓ Model class defined (instantiated fresh per fold)")

✓ Model class defined (instantiated fresh per fold)


##  OPTIMIZER, SCHEDULER, LOSS FUNCTIONS

In [8]:
criterion_species = nn.CrossEntropyLoss()
criterion_health  = nn.CrossEntropyLoss()
# scaler is reassigned as a global inside train_fold() for each fold
scaler = None
print("✓ Loss functions initialized")

✓ Loss functions initialized


## TRAINING UTILITIES

In [9]:
def accuracy(logits, targets):
    """Calculate accuracy from logits and targets"""
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()

def run_epoch(loader, model, optimizer=None, train=True, epoch=0):
    """Run one epoch of training or evaluation"""
    if train:
        model.train()
    else:
        model.eval()
    
    running = {
        "loss": 0.0,
        "acc_species": 0.0,
        "acc_health": 0.0,
        "acc_both": 0.0,
        "n": 0
    }
    total_batches = len(loader)
    
    for batch_idx, (imgs, y_species, y_health) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        y_species = y_species.to(device, non_blocking=True)
        y_health = y_health.to(device, non_blocking=True)
        
        with torch.set_grad_enabled(train):
            if USE_AMP and device.type == "cuda":
                with torch.amp.autocast('cuda'):
                    logits_species, logits_health = model(imgs)
                    loss = criterion_species(logits_species, y_species) + \
                           LOSS_WEIGHT_HEALTH * criterion_health(logits_health, y_health)
            else:
                logits_species, logits_health = model(imgs)
                loss = criterion_species(logits_species, y_species) + \
                       LOSS_WEIGHT_HEALTH * criterion_health(logits_health, y_health)
        
        if train:
            optimizer.zero_grad(set_to_none=True)
            if USE_AMP and device.type == "cuda":
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
        
        # Calculate accuracies
        acc_sp = accuracy(logits_species, y_species)
        acc_he = accuracy(logits_health, y_health)
        preds_sp = logits_species.argmax(1)
        preds_he = logits_health.argmax(1)
        acc_both = ((preds_sp == y_species) & (preds_he == y_health)).float().mean().item()
        
        # Update running statistics
        bs = imgs.size(0)
        running["loss"] += loss.item() * bs
        running["acc_species"] += acc_sp * bs
        running["acc_health"] += acc_he * bs
        running["acc_both"] += acc_both * bs
        running["n"] += bs
        
        # Print progress
        if (batch_idx + 1) % max(1, total_batches // 10) == 0 or (batch_idx + 1) == total_batches:
            avg_loss = running["loss"] / running["n"]
            avg_sp = running["acc_species"] / running["n"]
            avg_he = running["acc_health"] / running["n"]
            avg_both = running["acc_both"] / running["n"]
            print(f"  [{batch_idx + 1}/{total_batches}] loss: {avg_loss:.4f}, "
                  f"sp: {avg_sp:.3f}, he: {avg_he:.3f}, both: {avg_both:.3f}")
    
    # Calculate averages
    for k in ["loss", "acc_species", "acc_health", "acc_both"]:
        running[k] /= max(1, running["n"])
    
    return running

## TRAINING LOOP

In [10]:
import glob as _glob
import os

def find_latest_epoch_ckpt(fold_idx):
    """Return the fold_{fold_idx}_epoch_*.pt with the highest epoch number, or None."""
    files = _glob.glob(f"fold_{fold_idx}_epoch_*.pt")
    if not files:
        return None
    def extract_epoch(fn):
        try:
            return int(fn.split("_epoch_")[1].replace(".pt", ""))
        except Exception:
            return -1
    files.sort(key=extract_epoch)
    return files[-1]


def train_fold(fold_idx, train_samples, val_samples):
    """Train one fold with full checkpoint-resume support."""
    global scaler

    torch.manual_seed(SEED + fold_idx)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED + fold_idx)

    print(f"\n{'='*80}")
    print(f"FOLD {fold_idx} | Train: {len(train_samples)} | Val: {len(val_samples)}")
    print(f"{'='*80}")

    train_ds_fold = FlatSampleDataset(train_samples, transform=train_tf)
    val_ds_fold   = FlatSampleDataset(val_samples,   transform=eval_tf)
    train_loader_fold = DataLoader(
        train_ds_fold, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
    )
    val_loader_fold = DataLoader(
        val_ds_fold, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
    )

    model     = MultiTaskDeiTSmall(num_species=NUM_SPECIES, num_health=NUM_HEALTH,
                             pretrained=PRETRAINED, dropout=DROPOUT).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == "cuda")

    start_epoch       = 0
    best_metric       = 0.0
    epochs_no_improve = 0
    history = {
        "train_loss": [], "val_loss": [],
        "train_acc_species": [], "val_acc_species": [],
        "train_acc_health":  [], "val_acc_health":  [],
        "train_acc_both":    [], "val_acc_both":    []
    }

    resume_ckpt = find_latest_epoch_ckpt(fold_idx)
    if resume_ckpt is not None:
        print(f"Resuming from checkpoint: {resume_ckpt}")
        ckpt = torch.load(resume_ckpt, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch       = ckpt["epoch"] + 1
        best_metric       = ckpt["best_metric"]
        epochs_no_improve = ckpt["epochs_no_improve"]
        history           = ckpt["history"]
        print(f"Resumed from epoch {ckpt['epoch']+1}, best_metric={best_metric:.4f}")

    for epoch in range(start_epoch, EPOCHS):
        print(f"\nFold {fold_idx} | Epoch {epoch+1}/{EPOCHS} | LR: {optimizer.param_groups[0]['lr']:.2e}")

        train_stats = run_epoch(train_loader_fold, model, optimizer, train=True,  epoch=epoch)
        val_stats   = run_epoch(val_loader_fold,   model, optimizer=None, train=False, epoch=epoch)
        scheduler.step()

        history["train_loss"].append(train_stats["loss"])
        history["val_loss"].append(val_stats["loss"])
        history["train_acc_species"].append(train_stats["acc_species"])
        history["val_acc_species"].append(val_stats["acc_species"])
        history["train_acc_health"].append(train_stats["acc_health"])
        history["val_acc_health"].append(val_stats["acc_health"])
        history["train_acc_both"].append(train_stats["acc_both"])
        history["val_acc_both"].append(val_stats["acc_both"])

        print(f"  Train - Loss: {train_stats['loss']:.4f} | Sp: {train_stats['acc_species']:.3f} | He: {train_stats['acc_health']:.3f}")
        print(f"  Val   - Loss: {val_stats['loss']:.4f} | Sp: {val_stats['acc_species']:.3f} | He: {val_stats['acc_health']:.3f}")

        # Primary metric: average of the two task accuracies
        val_metric = (val_stats["acc_species"] + val_stats["acc_health"]) / 2.0

        if val_metric > best_metric:
            best_metric = val_metric
            epochs_no_improve = 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_metric": best_metric},
                       f"fold_{fold_idx}_best.pt")
            print(f"  ★ New best (val_metric={best_metric:.4f})")
        else:
            epochs_no_improve += 1
            print(f"  No improvement ({epochs_no_improve}/{PATIENCE})")

        # Save epoch checkpoint; delete the previous one to save disk
        prev_ckpt = find_latest_epoch_ckpt(fold_idx)
        epoch_ckpt_path = f"fold_{fold_idx}_epoch_{epoch}.pt"
        torch.save({
            "model":            model.state_dict(),
            "optimizer":        optimizer.state_dict(),
            "scheduler":        scheduler.state_dict(),
            "epoch":            epoch,
            "best_metric":      best_metric,
            "epochs_no_improve": epochs_no_improve,
            "history":          history
        }, epoch_ckpt_path)
        if prev_ckpt is not None and prev_ckpt != epoch_ckpt_path:
            try:
                os.remove(prev_ckpt)
            except FileNotFoundError:
                pass

        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping after {PATIENCE} epochs without improvement.")
            break

    # Load best checkpoint and evaluate on test set
    best_ckpt = torch.load(f"fold_{fold_idx}_best.pt", map_location=device)
    model.load_state_dict(best_ckpt["model"])
    model.eval()
    all_sp_preds, all_sp_true, all_he_preds, all_he_true = [], [], [], []
    with torch.no_grad():
        for imgs, y_sp, y_he in test_loader:
            imgs = imgs.to(device, non_blocking=True)
            logits_sp, logits_he = model(imgs)
            all_sp_preds.extend(logits_sp.argmax(1).cpu().numpy())
            all_sp_true.extend(y_sp.numpy())
            all_he_preds.extend(logits_he.argmax(1).cpu().numpy())
            all_he_true.extend(y_he.numpy())

    task_a_acc = accuracy_score(all_sp_true, all_sp_preds)
    task_b_acc = accuracy_score(all_he_true, all_he_preds)
    print(f"\nFold {fold_idx} Test — Species: {task_a_acc:.4f} | Health: {task_b_acc:.4f}")

    pd.DataFrame(history).to_csv(f"fold_{fold_idx}_history.csv", index=False)

    return {"task_a_accuracy": float(task_a_acc), "task_b_accuracy": float(task_b_acc)}

## COMPREHENSIVE TESTING FUNCTION

In [11]:
def comprehensive_test(model, test_loader, device, species_map, health_map):
    """
    Perform comprehensive testing with metrics and visualizations
    """
    model.eval()
    
    # Storage for predictions and ground truth
    all_species_preds = []
    all_species_true = []
    all_health_preds = []
    all_health_true = []
    all_both_correct = []
    
    print("Running comprehensive test evaluation...")
    
    with torch.no_grad():
        for batch_idx, (imgs, y_species, y_health) in enumerate(test_loader):
            imgs = imgs.to(device, non_blocking=True)
            y_species = y_species.to(device, non_blocking=True)
            y_health = y_health.to(device, non_blocking=True)
            
            # Get predictions
            logits_species, logits_health = model(imgs)
            preds_species = logits_species.argmax(dim=1)
            preds_health = logits_health.argmax(dim=1)
            
            # Store predictions and ground truth
            all_species_preds.extend(preds_species.cpu().numpy())
            all_species_true.extend(y_species.cpu().numpy())
            all_health_preds.extend(preds_health.cpu().numpy())
            all_health_true.extend(y_health.cpu().numpy())
            
            # Check if both predictions are correct
            both_correct = ((preds_species == y_species) & (preds_health == y_health)).cpu().numpy()
            all_both_correct.extend(both_correct)
    
    # Convert to numpy arrays
    all_species_preds = np.array(all_species_preds)
    all_species_true = np.array(all_species_true)
    all_health_preds = np.array(all_health_preds)
    all_health_true = np.array(all_health_true)
    all_both_correct = np.array(all_both_correct)
    
    # Reverse mapping for labels
    species_labels = {v: k.capitalize() for k, v in species_map.items()}
    health_labels = {v: k.capitalize() for k, v in health_map.items()}
    
    # -------------------------------
    # Print Metrics
    # -------------------------------
    print("\n" + "="*80)
    print("COMPREHENSIVE TEST RESULTS")
    print("="*80)
    
    # Overall accuracies
    species_acc = accuracy_score(all_species_true, all_species_preds)
    health_acc = accuracy_score(all_health_true, all_health_preds)
    both_acc = all_both_correct.mean()
    
    print(f"\n{'OVERALL ACCURACIES':^80}")
    print("-"*80)
    print(f"  Species Classification:  {species_acc:.4f} ({species_acc*100:.2f}%)")
    print(f"  Health Classification:   {health_acc:.4f} ({health_acc*100:.2f}%)")
    print(f"  Both Correct (Joint):    {both_acc:.4f} ({both_acc*100:.2f}%)")
    
    # Species Classification Report
    print(f"\n{'SPECIES CLASSIFICATION REPORT':^80}")
    print("-"*80)
    print(classification_report(
        all_species_true,
        all_species_preds,
        target_names=[species_labels[i] for i in sorted(species_labels.keys())],
        digits=4
    ))
    
    # Health Classification Report
    print(f"\n{'HEALTH/DISEASE CLASSIFICATION REPORT':^80}")
    print("-"*80)
    print(classification_report(
        all_health_true,
        all_health_preds,
        target_names=[health_labels[i] for i in sorted(health_labels.keys())],
        digits=4
    ))
    
    # Per-class joint accuracy
    print(f"\n{'PER-CLASS JOINT ACCURACY':^80}")
    print("-"*80)
    for sp_id in sorted(species_labels.keys()):
        for he_id in sorted(health_labels.keys()):
            # Find samples of this joint class
            mask = (all_species_true == sp_id) & (all_health_true == he_id)
            if mask.sum() > 0:
                joint_acc = all_both_correct[mask].mean()
                count = mask.sum()
                sp_name = species_labels[sp_id]
                he_name = health_labels[he_id]
                print(f"  {sp_name:8s} + {he_name:12s}: {joint_acc:.4f} "
                      f"({joint_acc*100:.2f}%) [{count:4d} samples]")
    
    # -------------------------------
    # Visualizations
    # -------------------------------
    
    # 1. Species Confusion Matrix
    fig, ax = plt.subplots(figsize=(8, 6))
    cm_species = confusion_matrix(all_species_true, all_species_preds)
    sns.heatmap(
        cm_species,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=ax,
        xticklabels=[species_labels[i] for i in sorted(species_labels.keys())],
        yticklabels=[species_labels[i] for i in sorted(species_labels.keys())]
    )
    ax.set_title(f'Species Classification\nAccuracy: {species_acc:.2%}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_species.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
    print(f"\nSaved species confusion matrix to 'confusion_matrix_species.png'")
    plt.close()
    
    # 2. Health Confusion Matrix
    fig, ax = plt.subplots(figsize=(8, 6))
    cm_health = confusion_matrix(all_health_true, all_health_preds)
    sns.heatmap(
        cm_health,
        annot=True,
        fmt='d',
        cmap='Greens',
        ax=ax,
        xticklabels=[health_labels[i] for i in sorted(health_labels.keys())],
        yticklabels=[health_labels[i] for i in sorted(health_labels.keys())]
    )
    ax.set_title(f'Health/Disease Classification\nAccuracy: {health_acc:.2%}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_health.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
    print(f"Saved health confusion matrix to 'confusion_matrix_health.png'")
    plt.close()
    
    # 3. Joint Accuracy Heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    joint_acc_matrix = np.zeros((len(species_labels), len(health_labels)))
    
    for sp_id in sorted(species_labels.keys()):
        for he_id in sorted(health_labels.keys()):
            mask = (all_species_true == sp_id) & (all_health_true == he_id)
            if mask.sum() > 0:
                joint_acc_matrix[sp_id, he_id] = all_both_correct[mask].mean()
    
    sns.heatmap(
        joint_acc_matrix,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        xticklabels=[health_labels[i] for i in sorted(health_labels.keys())],
        yticklabels=[species_labels[i] for i in sorted(species_labels.keys())],
        vmin=0,
        vmax=1,
        ax=ax,
        cbar_kws={'label': 'Accuracy'}
    )
    ax.set_title('Joint Classification Accuracy by Class', fontsize=12, fontweight='bold')
    ax.set_ylabel('Species')
    ax.set_xlabel('Health Status')
    plt.tight_layout()
    plt.savefig('joint_accuracy_heatmap.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
    print(f"Saved joint accuracy heatmap to 'joint_accuracy_heatmap.png'")
    plt.close()
    
    print("\n" + "="*80)
    print("Testing complete! Generated visualizations:")
    print("  - confusion_matrix_species.png")
    print("  - confusion_matrix_health.png")
    print("  - joint_accuracy_heatmap.png")
    print("="*80 + "\n")
    
    return {
        'species_accuracy': species_acc,
        'health_accuracy': health_acc,
        'joint_accuracy': both_acc,
        'species_preds': all_species_preds,
        'species_true': all_species_true,
        'health_preds': all_health_preds,
        'health_true': all_health_true
    }

## K-FOLD CROSS VALIDATION

In [12]:
def load_kfold_state():
    """Load k-fold training state from JSON file."""
    if Path(STATE_FILE).exists():
        with open(STATE_FILE, "r") as f:
            return json.load(f)
    return {"completed_folds": [], "fold_results": {}}

def save_kfold_state(state):
    """Save k-fold training state to JSON file."""
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_indices = list(range(len(all_samples)))

state = load_kfold_state()
print(f"K-fold state: completed folds = {state['completed_folds']}")
print(f"Starting {N_FOLDS}-fold stratified cross-validation...\n")

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(all_indices, strat_labels)):
    if fold_idx in state["completed_folds"]:
        print(f"Skipping fold {fold_idx} (already completed)")
        continue

    train_samples_fold = [all_samples[i] for i in train_idx]
    val_samples_fold   = [all_samples[i] for i in val_idx]

    fold_result = train_fold(fold_idx, train_samples_fold, val_samples_fold)

    state["completed_folds"].append(fold_idx)
    state["fold_results"][str(fold_idx)] = fold_result
    save_kfold_state(state)
    print(f"\nFold {fold_idx} completed and saved: {fold_result}")

print("\n" + "="*80)
print("ALL FOLDS COMPLETE")
print("="*80)

K-fold state: completed folds = []
Starting 5-fold stratified cross-validation...


FOLD 0 | Train: 5441 | Val: 1361
✓ Using DeiT-Small from timm library

Fold 0 | Epoch 1/50 | LR: 1.00e-04


RuntimeError: Numpy is not available

## COMPREHENSIVE TEST (BEST FOLD)

In [ ]:
state = load_kfold_state()
fold_results = state["fold_results"]

best_fold_idx = int(max(fold_results.keys(),
                        key=lambda k: fold_results[k]["task_a_accuracy"]))
print(f"Best fold: {best_fold_idx} "
      f"(species_acc={fold_results[str(best_fold_idx)]['task_a_accuracy']:.4f})")

best_model = MultiTaskDeiTSmall(num_species=NUM_SPECIES, num_health=NUM_HEALTH,
                          pretrained=PRETRAINED, dropout=DROPOUT).to(device)
ckpt = torch.load(f"fold_{best_fold_idx}_best.pt", map_location=device)
best_model.load_state_dict(ckpt["model"])
print(f"Loaded fold_{best_fold_idx}_best.pt")

print("\n" + "="*80)
print("Running Comprehensive Test Evaluation")
print("="*80 + "\n")

test_results = comprehensive_test(
    model=best_model, test_loader=test_loader, device=device,
    species_map=SPECIES_MAP, health_map=HEALTH_MAP
)

## SAMPLE PREDICTIONS (BEST FOLD)

In [ ]:
# Rebuild val loader for best fold using same StratifiedKFold split
skf_vis = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_splits_vis = list(skf_vis.split(list(range(len(all_samples))), strat_labels))
_, val_idx_best = fold_splits_vis[best_fold_idx]
val_samples_best = [all_samples[i] for i in val_idx_best]
val_ds_best = FlatSampleDataset(val_samples_best, transform=eval_tf)
val_loader_best = DataLoader(
    val_ds_best, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)

print("\n" + "="*80)
print("Generating Sample Predictions Visualization")
print("="*80 + "\n")

# Label mappings for visualization
species_labels = {
    0: 'Guava',
    1: 'Mango',
    2: 'Papaya'
}

health_labels = {
    0: 'Healthy',
    1: 'Anthracnose',
}

# Number of samples to display per class
amount = 10

# Collect samples from validation set
sample_images_by_class = {0: [], 1: [], 2: []}
sample_predictions_by_class = {0: [], 1: [], 2: []}
sample_ground_truth_by_class = {0: [], 1: [], 2: []}

best_model.eval()
with torch.no_grad():
    for images, species_batch, health_batch in val_loader_best:
        images = images.to(device)
        
        outputs = best_model(images)
        species_preds = outputs[0].argmax(1)
        health_preds = outputs[1].argmax(1)
        
        for i in range(len(images)):
            species_class = species_batch[i].item()
            
            if len(sample_images_by_class[species_class]) < amount:
                sample_images_by_class[species_class].append(images[i].cpu())
                sample_predictions_by_class[species_class].append({
                    'species': species_preds[i].item(),
                    'health': health_preds[i].item()
                })
                sample_ground_truth_by_class[species_class].append({
                    'species': species_batch[i].item(),
                    'health': health_batch[i].item()
                })
        
        if all(len(samples) >= amount for samples in sample_images_by_class.values()):
            break

# Visualize
mean = np.array(NORMALIZE_MEAN)
std = np.array(NORMALIZE_STD)

fig, axes = plt.subplots(NUM_SPECIES, amount, figsize=(3*amount, 9))

for row, species_idx in enumerate(sorted(sample_images_by_class.keys())):
    for col in range(amount):
        ax = axes[row, col]
        
        img = sample_images_by_class[species_idx][col]
        pred = sample_predictions_by_class[species_idx][col]
        gt = sample_ground_truth_by_class[species_idx][col]
        
        # Denormalize and display
        img_display = img.numpy().transpose(1, 2, 0)
        img_display = std * img_display + mean
        img_display = np.clip(img_display, 0, 1)
        
        ax.imshow(img_display)
        ax.axis('off')
        
        # Check correctness
        both_correct = (pred['species'] == gt['species']) and (pred['health'] == gt['health'])
        
        # Create title
        pred_sp = species_labels[pred['species']]
        pred_he = health_labels[pred['health']]
        gt_sp = species_labels[gt['species']]
        gt_he = health_labels[gt['health']]
        
        title = f"Pred: {pred_sp}, {pred_he}\nTrue: {gt_sp}, {gt_he}"
        color = 'green' if both_correct else 'red'
        ax.set_title(title, fontsize=8, color=color, fontweight='bold')
    
    # Add species label
    fig.text(0.02, 0.5 + (1 - row) * 0.3, species_labels[species_idx],
             fontsize=12, fontweight='bold', va='center', rotation=90)

plt.suptitle(f'Sample Predictions (Best Fold {best_fold_idx}) - {amount} Samples per Class\n(Green=Correct, Red=Incorrect)',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0.05, 0, 1, 0.99])
plt.savefig('sample_predictions.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
print(f"Saved sample predictions ({NUM_SPECIES*amount} total: {amount} per class) to 'sample_predictions.png'")
plt.close()


## FINAL SUMMARY

In [ ]:
state = load_kfold_state()
fold_results = state["fold_results"]

print("\n" + "="*80)
print("CROSS-VALIDATION RESULTS SUMMARY")
print("="*80)
print(f"\n{'Fold':>6} | {'Species Acc (Task A)':>22} | {'Health Acc (Task B)':>22}")
print("-" * 58)

task_a_accs, task_b_accs = [], []
for fold_idx in range(N_FOLDS):
    key = str(fold_idx)
    if key in fold_results:
        ta = fold_results[key]["task_a_accuracy"]
        tb = fold_results[key]["task_b_accuracy"]
        task_a_accs.append(ta)
        task_b_accs.append(tb)
        print(f"{fold_idx:>6} | {ta*100:>20.2f}% | {tb*100:>20.2f}%")

print("-" * 58)
print(f"\nTask A Accuracy : {np.mean(task_a_accs)*100:.2f} \u00b1 {np.std(task_a_accs)*100:.2f}")
print(f"Task B Accuracy : {np.mean(task_b_accs)*100:.2f} \u00b1 {np.std(task_b_accs)*100:.2f}")
print("="*80)